In [ ]:
from moabb.paradigms import P300
from moabb.datasets import *


dataset = BNCI2014_008()


paradigm = P300(
    resample=48,

)
cache_config = dict(
    use=True,
    save_raw=False,
    save_epochs=False,
    save_array=True,
    overwrite_raw=False,
    overwrite_epochs=False,
    overwrite_array=False,
)

In [ ]:
import tensorly as tl
from moabb.evaluations import WithinSessionEvaluation
from classification_erp import get_pipelines

if tl.get_backend() == 'cupy':
    n_jobs=1
else: 
    n_jobs=-1
    
evaluation = WithinSessionEvaluation(
    paradigm=paradigm,
    datasets=dataset,
    overwrite=False,
    random_state=42,
    n_jobs=1,
    suffix=f'bttda_dataset-{dataset.code}',
    cache_config=cache_config,
)
results =  evaluation.process(get_pipelines(n_jobs=n_jobs))

In [ ]:
results

In [ ]:
results.groupby(['dataset', 'pipeline'])['score'].aggregate('mean')

In [ ]:
results.groupby(['dataset', 'pipeline'])['score'].aggregate('mean').reset_index().groupby('pipeline')['score'].aggregate('mean')

In [ ]:
df_diff = results.pivot(index=['subject', 'session', 'channels', 'n_sessions', 'samples', 'dataset'], columns='pipeline', values='score')
df_diff = df_diff.reset_index()
df_diff['score_diff'] = df_diff['BTTDA'] - df_diff['HODA']
df_diff

In [ ]:
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'iframe'

def compare_score_plot(df, pipe1, pipe2):
    fig = px.scatter(df, x=pipe1, y=pipe2, color='dataset', facet_col='dataset', facet_col_wrap=5)
    fig.update_yaxes(scaleanchor="x")
    fig.update_xaxes(range=[.5, 1])
    fig.update_yaxes(range=[.5, 1])
    fig.add_shape(
        type="line",
        x0=0.5, y0=0.5, x1=1, y1=1,
        line=dict(color="gray", dash='dash'),
        layer="below" ,
        row='all', col='all', exclude_empty_subplots=True
    )
    

    return fig

fig = compare_score_plot(df_diff, 'HODA', 'BTTDA')
fig.update_layout(
    autosize=False,
    width=1800,
    height=1800,
)
fig.update_layout(showlegend=False)
fig